# Gateway Control-Plane Theory

## The brief: Riverside House has three model deployments and no safe way to choose among them

The editing assistant now has a hosted fast route, a hosted premium route, and a local confidential-data route. The application can call all three, but every call site knows different request fields, response shapes, prices, regions, and failure rules. A cheap route can be illegal. A retry can quietly become duplicate spend. A cache hit can be stale.

**Guiding question:** what must happen between an application request and a model deployment so the choice is safe, bounded, and observable?

> **What you finished last time:** evaluation told you whether an answer was acceptable.
> **What this notebook delivers:** a 45-60 minute failure-first mental model for the gateway control plane.
> **Prerequisite for the next notebook:** [Gateway Routing and Resilience Lab](02-gateway-routing-resilience-lab.ipynb) rebuilds every measured mechanism from a fresh kernel. Kernels do not share memory; the lab repeats only the small Riverside fixtures it needs.

| Part | Complaint that forces it | You will leave with |
|---|---|---|
| 1. Boundary | "Why not put this inside the model server?" | Control plane versus inference server |
| 2. Contract | "Every provider shape leaks into application code" | One normalized request/response |
| 3. Eligibility | "The optimizer picked an illegal route" | Filter first, optimize second |
| 4. Lifecycle | "Correct pieces in the wrong order still fail" | Cache, limits, budget, route, settle |
| 5. Recovery | "Retry and fallback are being treated as synonyms" | Same-route retry versus route-changing fallback |

**Scope map**

- **Built and measured here:** response-shape normalization, eligibility-before-routing, ordering consequences, and a deterministic retry/fallback trace.
- **Illustrated here; built in the lab:** caches, request/token/concurrency limits, atomic budget reserve/settle, routing strategies, and telemetry.
- **Forward boundary:** circuit breakers and serving backpressure stay named but unmeasured. Continuous batching, KV-cache management, and GPU scheduling belong to the inference-serving layer.

## 1. A gateway is a request control plane, not an inference server

You already have deployments that can generate text. The missing system decides **whether**, **where**, and **under what bounded policy** a request may run.

\`\`\`mermaid
flowchart LR
    A[Application] --> B[Gateway contract]
    B --> C[Policy and controls]
    C --> D[Eligible routes]
    D --> E[Provider adapter]
    E --> F[Model deployment]
    F --> G[Normalized result]
    classDef input fill:#1e3a8a,color:#fff,stroke:#1e3a8a;
    classDef process fill:#1d4ed8,color:#fff,stroke:#1d4ed8;
    classDef caution fill:#b45309,color:#fff,stroke:#b45309;
    classDef output fill:#15803d,color:#fff,stroke:#15803d;
    class A input;
    class B,C,E process;
    class D caution;
    class F,G output;
\`\`\`

The gateway controls traffic **to** deployments. The deployment still owns token generation, batching, GPU scheduling, and its own queues. That boundary matters: a gateway can reject overload before dispatch, but it cannot repair a saturated GPU scheduler by renaming the queue "routing."

![Handwritten map of the request control plane](images/01-llm-gateway-theory-01.png)

**Predict:** if the local deployment queue grows, which layer can choose a different eligible deployment? Which layer can pack two decode steps into one GPU batch?

## 2. Attempt 1: let the application call every provider directly

Three providers all accept "a prompt." That similarity lasts until you inspect the response.

In [ ]:
# Part 2 - Expose the provider-shape leak
prompt = "How many chapters are in The Cartographer's Cipher?"

openai_native = {
    "choices": [{"message": {"content": "There are 23 chapters."}}],
    "usage": {"prompt_tokens": 9, "completion_tokens": 6},
}
anthropic_native = {
    "content": [{"text": "There are 23 chapters."}],
    "usage": {"input_tokens": 9, "output_tokens": 6},
}
local_native = {
    "generated_text": "There are 23 chapters.",
    "token_count": 15,
}

extractors = {
    "openai-shaped": lambda r: (r["choices"][0]["message"]["content"], r["usage"]["prompt_tokens"] + r["usage"]["completion_tokens"]),
    "anthropic-shaped": lambda r: (r["content"][0]["text"], r["usage"]["input_tokens"] + r["usage"]["output_tokens"]),
    "local-shaped": lambda r: (r["generated_text"], r["token_count"]),
}

for name, native in zip(extractors, [openai_native, anthropic_native, local_native]):
    text, tokens = extractors[name](native)
    print(f"{name:<18} text={text!r}, tokens={tokens}")

print("\nComplaint: one application question required three extraction paths.")
print("A provider swap is now an application-code change, not a deployment decision.")

### The smallest fix: normalize once at the edge

\`\`\`mermaid
flowchart LR
    A[Provider A shape] --> D[Adapter A]
    B[Provider B shape] --> E[Adapter B]
    C[Local shape] --> F[Adapter C]
    D --> G[GatewayResponse]
    E --> G
    F --> G
    classDef source fill:#b45309,color:#fff,stroke:#b45309;
    classDef adapter fill:#1d4ed8,color:#fff,stroke:#1d4ed8;
    classDef contract fill:#15803d,color:#fff,stroke:#15803d;
    class A,B,C source;
    class D,E,F adapter;
    class G contract;
\`\`\`

The normalized response must keep enough evidence to reconstruct the decision:

| Field | Why it survives normalization |
|---|---|
| text and finish status | Stable caller behavior |
| provider, deployment, pinned model version | Attribution and rollback |
| input/output tokens | Token limits and settlement |
| latency and cost | SLO and budget evidence |
| request ID and attempt number | One caller result, many internal attempts |
| normalized error code and retryable flag | Recovery policy without SDK-specific exceptions |

Do not normalize away the route that actually answered. "The gateway succeeded" is useless when one fallback deployment is serving every request.

In [ ]:
# Part 2 - Prove three native shapes can satisfy one response contract
def adapt_openai(native):
    usage = native["usage"]
    return {
        "text": native["choices"][0]["message"]["content"],
        "input_tokens": usage["prompt_tokens"],
        "output_tokens": usage["completion_tokens"],
    }


def adapt_anthropic(native):
    usage = native["usage"]
    return {
        "text": native["content"][0]["text"],
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
    }


def adapt_local(native):
    return {
        "text": native["generated_text"],
        "input_tokens": 9,
        "output_tokens": native["token_count"] - 9,
    }


normalized = [
    adapt_openai(openai_native),
    adapt_anthropic(anthropic_native),
    adapt_local(local_native),
]
contract_keys = set(normalized[0])
same_contract = all(set(item) == contract_keys for item in normalized)
same_answer = len({item["text"] for item in normalized}) == 1

print(f"same normalized keys: {same_contract}")
print(f"same answer text:     {same_answer}")
print(f"contract keys:        {sorted(contract_keys)}")
print("Payoff: application code reads one shape; adapters contain provider variation.")

## 3. Attempt 2: route first, then check policy

Riverside's cheapest route is hosted in another region. It is fine for public marketing copy and forbidden for unpublished manuscripts.

**Predict:** a cost router sees prices of $0.00005, $0.00015, and $0.00080 per 1K tokens. Which route wins if policy runs after optimization?

In [ ]:
# Part 3 - Measure why eligibility must precede optimization
routes = [
    {"name": "external-cheap", "cost_per_1k": 0.00005, "region": "us", "allows_confidential": False, "healthy": True},
    {"name": "hosted-approved", "cost_per_1k": 0.00015, "region": "uk", "allows_confidential": True, "healthy": True},
    {"name": "local-private", "cost_per_1k": 0.0, "region": "on-prem", "allows_confidential": True, "healthy": True},
]
request = {"region": "uk", "confidential": True, "requires_hosted": True}

wrong_choice = min(routes, key=lambda route: route["cost_per_1k"])
eligible = [
    route
    for route in routes
    if route["healthy"]
    and (not request["confidential"] or route["allows_confidential"])
    and (not request["requires_hosted"] or route["region"] == request["region"])
]
right_choice = min(eligible, key=lambda route: route["cost_per_1k"])

print(f"optimize before policy: {wrong_choice['name']}")
print(f"eligible set:           {[route['name'] for route in eligible]}")
print(f"policy then optimize:   {right_choice['name']}")
print("\nComplaint: the globally cheapest route was not a legal candidate.")
print("Rule: eligibility is a filter; routing is an optimization inside that filtered set.")

\`\`\`mermaid
flowchart LR
    A[Normalized request] --> B{Eligibility}
    B -->|capability region data policy health| C[Approved set]
    B -->|fails any hard rule| D[Excluded]
    C --> E{Routing objective}
    E --> F[Least busy]
    E --> G[Lowest cost]
    E --> H[Weighted or latency aware]
    classDef input fill:#1e3a8a,color:#fff,stroke:#1e3a8a;
    classDef policy fill:#b45309,color:#fff,stroke:#b45309;
    classDef reject fill:#b91c1c,color:#fff,stroke:#b91c1c;
    classDef output fill:#15803d,color:#fff,stroke:#15803d;
    class A input;
    class B,E policy;
    class D reject;
    class C,F,G,H output;
\`\`\`

Latency, cost, weights, and in-flight counts are **soft objectives**. Capability, context length, region, tenant policy, data sensitivity, and health are **hard constraints**. A clever optimizer must never turn a hard constraint into a score it can trade away.

## 4. Attempt 3: keep all controls, but run them in a convenient order

A gateway can contain the right mechanisms and still charge a cache hit, consume a concurrency slot before it knows work is needed, or spend before it reserves.

\`\`\`mermaid
flowchart LR
    A[Authenticate and normalize] --> B[Versioned exact cache]
    B -->|hit| H[Output policy and return]
    B -->|miss| C[Estimate tokens]
    C --> D[Request token concurrency limits]
    D --> E[Reserve budget]
    E --> F[Eligibility then routing]
    F --> G[Dispatch recover normalize]
    G --> I[Settle actual cost]
    I --> J[Output policy cache telemetry]
    J --> H
    classDef input fill:#1e3a8a,color:#fff,stroke:#1e3a8a;
    classDef process fill:#1d4ed8,color:#fff,stroke:#1d4ed8;
    classDef caution fill:#b45309,color:#fff,stroke:#b45309;
    classDef output fill:#15803d,color:#fff,stroke:#15803d;
    class A input;
    class B,C,D,F,G,I,J process;
    class E caution;
    class H output;
\`\`\`

The cache key must include tenant, normalized input, pinned model, generation parameters, corpus version, policy version, and relevant tool state. A hit still passes current output policy. Sensitive prompts are not cached by default.

In [ ]:
# Part 4 - Measure the cost of putting limits and budget before the cache
traffic = ["chapter-count", "chapter-count", "character-list", "chapter-count"]
cache = {"chapter-count"}
estimated_cost = 0.002

wrong_budget_reserved = len(traffic) * estimated_cost
wrong_limit_slots = len(traffic)

misses = [prompt for prompt in traffic if prompt not in cache]
right_budget_reserved = len(misses) * estimated_cost
right_limit_slots = len(misses)

print(f"requests:                         {len(traffic)}")
print(f"cache misses requiring dispatch: {len(misses)}")
print(f"budget reserved before cache:    USD {wrong_budget_reserved:.3f}")
print(f"budget reserved after cache:     USD {right_budget_reserved:.3f}")
print(f"limit slots before cache:        {wrong_limit_slots}")
print(f"limit slots after cache:         {right_limit_slots}")
print("\nPayoff: safe cache hits avoid provider-facing limits and spend reservations.")
print("Boundary: authentication and current output policy still apply to hits.")

### Budget is a ledger, not a request counter

A request limit cannot promise a spending limit because routes have different prices and outputs have different lengths.

1. Estimate tokens and cost.
2. Atomically reserve the estimate before dispatch.
3. Reject predictably if the reservation would cross the ceiling.
4. Settle the reservation against actual usage.
5. Attribute retries and failed attempts too; failure is not automatically free.

The lab measures both refund and overage settlement. The theory rule is simpler: **reserve before uncertain work, settle after observed work**.

## 5. Attempt 4: call every second attempt a retry

A retry calls the **same deployment again** because the error may be transient. A fallback changes to a **different eligible deployment** because the current route has exhausted its bounded chance.

\`\`\`mermaid
flowchart LR
    A[Attempt 1 on Route A] --> B{Error retryable?}
    B -->|no| C[Return normalized failure]
    B -->|yes and deadline remains| D[Retry Route A]
    D --> E{Succeeded?}
    E -->|yes| F[Return result]
    E -->|no retry budget exhausted| G[Fallback to eligible Route B]
    G --> H[Return result or clear failure]
    classDef input fill:#1e3a8a,color:#fff,stroke:#1e3a8a;
    classDef process fill:#1d4ed8,color:#fff,stroke:#1d4ed8;
    classDef caution fill:#b45309,color:#fff,stroke:#b45309;
    classDef reject fill:#b91c1c,color:#fff,stroke:#b91c1c;
    classDef output fill:#15803d,color:#fff,stroke:#15803d;
    class A input;
    class D,G process;
    class B,E caution;
    class C reject;
    class F,H output;
\`\`\`

Retry transient timeouts, selected throttling, connection failures, and selected server errors. Do not retry invalid input, policy denial, or deterministic context overflow. Use a small attempt limit, per-attempt timeouts, jittered backoff, and one overall deadline.

![Static recovery storyboard: same-route retry before route-changing fallback](images/01-llm-gateway-theory-02.png)

The historical storyboard labels retry/backoff and reserve/settle as unmeasured in the old notebook. The companion lab now measures both; the picture remains useful for the control-flow distinction.

In [ ]:
# Part 5 - Animate a deterministic attempt trace derived from recorded state
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

attempt_trace = [
    {"step": 1, "route": "hosted-fast", "event": "timeout", "kind": "initial"},
    {"step": 2, "route": "hosted-fast", "event": "timeout", "kind": "retry"},
    {"step": 3, "route": "hosted-premium", "event": "success", "kind": "fallback"},
]

colors = {"initial": "#1d4ed8", "retry": "#b45309", "fallback": "#15803d"}
fig, axis = plt.subplots(figsize=(8, 3.2))
axis.set_xlim(0.5, 3.5)
axis.set_ylim(0, 1)
axis.set_xticks([1, 2, 3])
axis.set_xticklabels(["initial", "same-route retry", "different-route fallback"])
axis.set_yticks([])
bars = axis.bar([1, 2, 3], [0, 0, 0], color=[colors[item["kind"]] for item in attempt_trace])
title = axis.set_title("")


def update_attempt(frame):
    for index, bar in enumerate(bars):
        bar.set_height(0.72 if index <= frame else 0)
    current = attempt_trace[frame]
    title.set_text(
        f"Step {current['step']}: {current['route']} -> {current['event']}"
    )
    return [*bars, title]


animation = FuncAnimation(fig, update_attempt, frames=len(attempt_trace), interval=700, blit=False)
plt.close(fig)
print("Animation: watch the route label. Step 2 stays on the same route; step 3 changes route.")
display(HTML(animation.to_jshtml(fps=2)))

route_changes = sum(
    attempt_trace[index]["route"] != attempt_trace[index - 1]["route"]
    for index in range(1, len(attempt_trace))
)
print(f"recorded attempts: {len(attempt_trace)}")
print(f"route changes:     {route_changes}")
print("Measured distinction: the retry added work without changing route; fallback changed route once.")

### One caller result still needs an attempt story

The caller receives one normalized result. Operators need every internal attempt:

| Scope | Minimum telemetry |
|---|---|
| Request | request ID, tenant, cache result, limit decision, budget reservation, final status |
| Attempt | route, model version, attempt kind, normalized error, retryable flag, tokens, latency, cost |
| Outcome | route that answered, retry count, fallback position, settled cost, remaining deadline |

Redact prompts, secrets, and personal identifiers. Track p95/p99 latency, time to first token for streaming, cache hit rate, rejection reason, retry amplification, fallback share, cost per acceptable result, and quality by route. Transport success is not answer quality.

## Toy-to-real bridge

| Teaching mechanism | Production counterpart | What changes | What does not change |
|---|---|---|---|
| Python dictionaries | Typed gateway schema / OpenAPI contract | Validation and versioning become strict | Callers see one contract |
| List filtering | Policy engine and deployment registry | State becomes distributed and dynamic | Hard constraints run before optimization |
| In-memory cache set | Tenant-isolated shared cache | TTL, invalidation, encryption, and policy checks | A complete versioned key decides reuse |
| Scalar reservation | Atomic ledger transaction | Concurrency, persistence, and reconciliation | Reserve before work; settle actual usage |
| Three recorded attempts | Deadline-aware retry policy | Backoff, timeouts, SDK error mapping | Retry stays; fallback changes route |
| Printed trace | Metrics, logs, and distributed traces | Sampling and redaction matter | Request and attempt stories remain separate |

The local alias \`local-llama-8b\` is routing metadata retained from the original lesson. The deterministic lab maps it to \`HuggingFaceTB/SmolLM2-135M-Instruct\` and loads no transformer. The alias is not evidence that an 8B model ran.

In [ ]:
# Closing scorecard - Answer the opening questions with measured notebook evidence
scorecard = {
    "native response shapes exposed": len(extractors),
    "normalized contracts identical": same_contract,
    "ineligible cheap route blocked": wrong_choice["name"] not in [route["name"] for route in eligible],
    "dispatches avoided by cache-first ordering": len(traffic) - len(misses),
    "retry/fallback route changes": route_changes,
}

for claim, value in scorecard.items():
    print(f"{claim:<48} {value}")

print("\nDecision: normalize first, filter eligibility before routing, check a complete")
print("cache before provider-facing controls, reserve before dispatch, settle after usage,")
print("and keep retry distinct from fallback in both policy and telemetry.")

## Coverage ledger and handoff

### Built and measured
- Three incompatible provider response shapes normalized to one contract.
- A cheapest-but-illegal route removed before cost optimization.
- Cache-first ordering avoided unnecessary limit slots and budget reservations.
- A recorded failure chain separated same-route retry from route-changing fallback.

### Explained or illustrated
- Complete lifecycle ordering, response/error taxonomy, exact cache key dimensions, request/attempt telemetry, and reserve/settle semantics.
- Rate, token, and concurrency controls; round-robin, least-busy, weighted, latency-aware, and cost-aware routing.

### Named and intentionally out of scope
- Circuit breakers remain a forward reliability boundary until measured.
- Gateway admission can reject overload; serving backpressure, continuous batching, KV-cache management, and GPU scheduling remain inside the inference-serving track.
- Authentication, managed identity, networking, Azure policy behavior, provider reliability, answer quality, and savings need live validation.

**Key insights to keep**

1. A gateway is the request control plane around deployments, not the token-generation engine.
2. Normalize provider variation once; preserve attribution and error semantics.
3. Eligibility removes illegal choices. Routing optimizes only the approved set.
4. Correct controls in the wrong order still waste quota, spend, and concurrency.
5. Retry stays on a route; fallback changes route without broadening policy.

The unresolved question is now measurable: **which policies actually improve Riverside's success, latency, and cost without hiding unsafe reuse or rejected work?** Continue in [Gateway Routing and Resilience Lab](02-gateway-routing-resilience-lab.ipynb).